# RAGOptimizer - Week 6 Code Companion
**Applied GenAI & Agentic AI Engineering Course · Week 6**

This notebook walks through the RAGOptimizer service - HyDE query transformation, cross-encoder reranking, and extractive context compression wired on top of the Week 5 CitationRAG service.

Each module is shown two ways:
- **Direct import** - call the function, see the output, understand what it does
- **HTTP endpoint** - hit the live FastAPI service the same way a client would

---
### Before you start
1. Server running: `uvicorn app.main:app --reload` (from the `root` directory)
2. `.env` filled in - copy `.env.example` → `.env` and add your `OPENAI_API_KEY`
3. Dependencies installed: `pip install -r requirements.txt`
4. Browser UI available at: `http://localhost:8000` (open in a separate tab)
5. Run the **Setup** cell below once.

> **Windows note:** All curl cells use `%%cmd` with `\"` to escape inner quotes. Single quotes are not supported in Windows cmd.

> **Platform note.** Cells written with the `%%cmd` magic run on Windows only. On macOS or Linux, run the same command in a terminal, replacing the line-continuation caret `^` with a backslash `\`. Every `%%cmd` cell in this notebook has a Python equivalent beside it, so nothing here is Windows-only in substance.

> **Outputs are cleared on purpose.** Run the cells yourself. The numbers you get are the numbers to report, and a committed output from somebody else's machine is exactly the kind of evidence this week teaches you not to trust.


In [ ]:
# Setup -- run this cell first
import requests, json, textwrap

BASE = 'http://localhost:8000'

# Demo query used throughout the notebook.
# The index is the Week 4/5 corpus - the 'Attention Is All You Need' paper -
# so the demo query must be answerable FROM THAT PAPER. (The JWT-rotation example
# in the concept video is illustrative, drawn from a runbook corpus, not this index.)
DEMO_QUERY = 'what is scaled dot-product attention'

# RECIPIENT = 'you@example.com'  # no email endpoint this week

print('Setup complete.')
print('BASE:', BASE)

---
## 1 · Health Check - `GET /health`
Confirms the server is alive and shows which model is loaded.

> This section is identical across all weeks. Do not modify it.

In [ ]:
%%cmd
curl -s http://localhost:8000/health

In [ ]:
# Health check - Python
r = requests.get(f'{BASE}/health')
print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2))
print()
data = r.json()
print('Main model :', data.get('models', {}).get('main', '?'))
print('HyDE model :', data.get('models', {}).get('hyde', '?'))

---
## 2 · Browser UI - `GET /`

The server now serves a browser demo UI at `GET /`. Open it in a separate tab to interact with the pipeline visually.

The UI provides:
- A **Query** textarea with four demo pills from the Attention paper (scaled dot-product attention, multi-head attention, positional encoding, encoder-decoder)
- **Pipeline status chips** - auto-loaded from `/config`, show HyDE / Reranker / Compressor state
- **Ask RAGOptimizer** - POST /ask with a rendered answer card, citations, and full pipeline trace
- **Compare Pipelines** - POST /eval/compare, runs all 10 golden questions through both the W5 baseline and the full W6 pipeline and renders a delta table
- **📖 README** button in the header - opens the rendered README at `GET /readme`

> **Intentional deviation:** no SSE streaming in the UI - `/ask` returns full JSON. The HyDE + rerank + compress pipeline does not produce token-by-token output.


In [ ]:
from IPython.display import display, HTML
display(HTML('<a href="http://localhost:8000" target="_blank" style="font-size:15px">'
             'Open Browser UI: http://localhost:8000</a>'))
display(HTML('<a href="http://localhost:8000/readme" target="_blank" style="font-size:15px">'
             'Open README: http://localhost:8000/readme</a>'))

In [ ]:
# Verify / and /readme routes respond
r_ui     = requests.get(f'{BASE}/')
r_readme = requests.get(f'{BASE}/readme')
print('GET /       status:', r_ui.status_code,     ' content-type:', r_ui.headers.get('content-type', '?'))
print('GET /readme status:', r_readme.status_code, ' content-type:', r_readme.headers.get('content-type', '?'))

---
## 3 · Active Config - `GET /config`

Surfaces every tunable toggle so you can confirm what is wired before running a benchmark.
This is the first thing to check when results look wrong - is HyDE on? Which reranker? What's the keep fraction?

Key concept: **settings via `pydantic-settings`** - all config lives in one typed `Settings` object, cached with `lru_cache`. Nothing in business code touches `os.environ` directly.

Response shape:
```json
{
  "model_id": "gpt-5.4-mini-2026-03-17",
  "hyde_model": "gpt-5.4-nano-2026-03-17",
  "hyde_enabled": true,
  "reranker_model": "BAAI/bge-reranker-base",
  "compressor_enabled": true,
  "vector_top_k_wide": 30,
  "vector_top_k_narrow": 5
}
```

In [ ]:
%%cmd
curl -s http://localhost:8000/config

In [ ]:
r = requests.get(f'{BASE}/config')
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    cfg = r.json()
    print('Active configuration:')
    print(json.dumps(cfg, indent=2))
    print()
    print('Main model       :', cfg['model_id'])
    print('HyDE probe model :', cfg['hyde_model'])
    print('HyDE enabled     :', cfg['hyde_enabled'])
    print('Reranker         :', cfg['reranker_model'])
    print('Compressor on    :', cfg['compressor_enabled'])
    print('Retrieve wide/narrow:', cfg['vector_top_k_wide'], '/', cfg['vector_top_k_narrow'])

---
## 4 · HyDE Module - `app/hyde.py`

**HyDE = Hypothetical Document Embeddings.** Instead of embedding the user's question, we ask the model to write a plausible answer first, then embed *that*. The content is discarded - only the embedding shape matters. This closes the vocabulary gap between how users phrase questions and how documentation is written.

```
User query: "how do I rotate the JWT signing key?"
                    ↓  hyde_probe()
Hypothetical answer: "To rotate the JWT signing key, access the secrets manager..."
                    ↓  embed()
Retrieval probe vector  →  vector index top-30
```

**Why `gpt-5.4-nano` here?** We use the small/fast model for the probe - the content gets thrown away after embedding, so generation quality matters less than latency. The main pipeline uses `gpt-5.4-mini-2026-03-17` for answers.

> Requires `OPENAI_API_KEY` in `.env` - this cell makes a live API call.

In [ ]:
# Demo: call hyde_probe directly
import sys, os
sys.path.insert(0, os.getcwd())  # make sure app/ is importable

from app.hyde import hyde_probe

probe = hyde_probe(DEMO_QUERY)
print('Query  :', DEMO_QUERY)
print()
print('Hypothetical answer (will be embedded, not shown to user):')
print(textwrap.fill(probe, width=80))

In [ ]:
# Run the same query twice - notice vocabulary shifts each time.
# That variance is fine: we union HyDE candidates with the original query candidates.
probe_a = hyde_probe(DEMO_QUERY)
probe_b = hyde_probe(DEMO_QUERY)

print('Run A:', textwrap.fill(probe_a, width=80))
print()
print('Run B:', textwrap.fill(probe_b, width=80))

---
## 5 · Reranker Module - `app/reranker.py`

The vector index returns top-30 by approximate nearest-neighbour - high recall, mid precision. The cross-encoder reranker scores every `(query, chunk)` pair directly: slower but far more accurate. We keep the top-5 after sorting.

**Why batch?** Calling the cross-encoder one pair at a time drives p95 latency over the SLA. Batching all 30 pairs as a single tensor keeps it under 110ms.

```python
# Bad (before fix): one forward pass per pair
for pair in pairs:
    score = reranker.predict([pair])

# Good (after fix): one forward pass for all 30
scores = reranker.predict(pairs, batch_size=30)
```

> This cell lazy-loads `BAAI/bge-reranker-base` from HuggingFace on first run (~1 GB download). Subsequent runs use the cached model.

In [ ]:
from app.reranker import rerank
from app.schemas import Chunk

# Build 5 sample chunks to rerank
sample_chunks = [
    Chunk(chunk_id='c1', text='[c1]\nJWT tokens expire after 24h. Refresh by calling /auth/refresh.', score=0.7),
    Chunk(chunk_id='c2', text='[c2]\nThe JWT signing key lives in the secrets manager under jwt/signing-key.', score=0.65),
    Chunk(chunk_id='c3', text='[c3]\nLog rotation runs nightly. Set LOG_RETENTION_DAYS in config.', score=0.62),
    Chunk(chunk_id='c4', text='[c4]\nTo rotate the JWT signing key: 1) generate new key in secrets manager 2) deploy with overlap window.', score=0.60),
    Chunk(chunk_id='c5', text='[c5]\nDatabase backups are encrypted with AES-256.', score=0.55),
]

print('Before rerank (vector scores):')
for c in sample_chunks:
    print(f'  {c.chunk_id}  score={c.score:.3f}  "{c.text[:60]}..."')

reranked = rerank(DEMO_QUERY, sample_chunks)

print()
print('After rerank (cross-encoder scores, descending):')
for c in reranked:
    print(f'  {c.chunk_id}  score={c.score:.4f}  "{c.text[:60]}..."')

Notice that `c4` - the most specific JWT *rotation* chunk - jumps to the top even though it had the lowest vector score. That's the precision gain from the cross-encoder.

---
## 6 · Compressor Module - `app/compressor.py`

After reranking we have the right 5 chunks - but each chunk may contain sentences irrelevant to *this specific query*. The compressor keeps only the top-fraction of sentences by lexical overlap score, reducing context tokens without losing grounding.

**The invariant:** the first line of every chunk (the `[chunk-id]` marker) is *always* preserved. The citation validator in `main.py` depends on it. Only body sentences are eligible for dropping.

```
[runbook-jwt-042]          ← ALWAYS kept (anchored outside scoring)
An apple a day...          ← scored: 0.00  → dropped
JWT signing key in secrets ← scored: 0.67  → kept
Cats are good pets.        ← scored: 0.00  → dropped
Cutover during overlap...  ← scored: 0.20  → kept (if keep_fraction allows)
```

In [ ]:
from app.compressor import compress_chunk
from app.schemas import Chunk

chunk = Chunk(
    chunk_id='runbook-jwt-rotation-042',
    text=(
        '[runbook-jwt-rotation-042]\n'
        'An apple a day keeps the doctor away. '
        'The JWT signing key lives in the secrets manager. '
        'Cats are good pets. '
        'Cutover happens during the overlap window.'
    ),
    source='runbook.md',
)

compressed = compress_chunk(chunk, query=DEMO_QUERY)

print('Original text:')
print(chunk.text)
print()
print('Compressed text:')
print(compressed.text)
print()
print(f'Words saved (approx): {len(chunk.text.split()) - len(compressed.text.split())}')
print('Marker preserved      :', compressed.marker_line == '[runbook-jwt-rotation-042]')

---
## 7 · Full Pipeline - `app/retriever.py`

`retriever.py` is the composer. It knows about all four pieces and wires them in order:

```
query
  ↓
hyde_probe() → _dense_search()   →  HyDE candidates (top-30, ANN only)
_hybrid_search()                 →  Original query candidates (top-30, dense+BM25+RRF)
_union_dedupe()                  →  Up to 30 unique candidates
rerank()                         →  Sorted by cross-encoder score
[:top_k]                         →  Top-5
compress()                       →  Sentence-level trimming, markers preserved
  ↓
list[Chunk]  (same type as W5 output)
```

Everything downstream - the `/ask` route, the citation prompt, the eval harness - sees the same `list[Chunk]` it always did from Week 5. Only the retrieval path changed.

> **Note:** retrieval runs against the Week 4 KnowledgeVault Qdrant collection. Set `QDRANT_URL`, `QDRANT_API_KEY`, and `QDRANT_COLLECTION` in `.env` (copy from Week 4) - with the collection populated you'll see 5 compressed chunks below.


In [ ]:
from app.retriever import retrieve

# Runs the real pipeline against the W4 Qdrant collection (QDRANT_* in .env).
chunks = retrieve(DEMO_QUERY)

print(f'retrieve("{DEMO_QUERY}")')
print(f'  returned {len(chunks)} chunk(s)')
print()
if chunks:
    for c in chunks:
        print(f'  [{c.chunk_id}]  score={c.score:.4f}')
        print(f'  {c.text[:120]}...')
        print()
else:
    print('  (no candidates - check QDRANT_URL / QDRANT_COLLECTION in .env and that the W4 index is populated)')


---
## 8 · Ask Endpoint - `POST /ask`

The `/ask` route is the same shape as Week 5's CitationRAG - same request body, same response schema. Only the retrieval path behind it changed.

The citation prompt and validator in `app/citation.py` are fully implemented - the system prompt enforces the six citation rules and `_validate_citations` drops any hallucinated chunk-IDs before the response leaves the service.

Request body:
```json
{ "query": "...", "conversation_history": [] }
```

Response shape:
```json
{
  "answer": "...",
  "citations": [ { "chunk_id": "...", "quote": "..." } ],
  "confidence": "high",
  "fallback_triggered": false,
  "pipeline_trace": { "...": "full trace of every stage" }
}
```

> If retrieval returns no candidates (Qdrant not configured or collection empty), `/ask` returns **503** instead of hallucinating an answer.


In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/ask -H "Content-Type: application/json" -d "{\"query\": \"what is scaled dot-product attention\"}"

In [ ]:
r = requests.post(f'{BASE}/ask', json={'query': DEMO_QUERY})
print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2))

---
## 9 · Evaluation - `POST /eval`

The headline endpoint pair starts here. `/eval` runs all 10 golden questions (`data/golden_dataset.json`) through the **full W6 pipeline** and returns a five-metric dashboard plus per-row detail with pipeline traces.

**Honest read:** these are deterministic proxies, each a pass-rate over all 10 rows - one row moves any metric by exactly 0.10. `citation_validity` is a validator-backed invariant (the validator strips invalid IDs upstream, so the metric confirms the invariant held). `groundedness` counts correct refusals as grounded. Statistical rigor arrives in Week 7.

Expected response shape:
```json
{
  "groundedness": 1.0,
  "citation_validity": 1.0,
  "citation_recall": 1.0,
  "false_answer_rate": 0.0,
  "false_refusal_rate": 0.0,
  "rows_scored": 10,
  "rows": [ { "question": "...", "expected": "answer", "actual": "answered", "passed": true, "...": "..." } ]
}
```

> Runs the full pipeline 10 times - expect **60-120 s** and real API spend.


In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/eval


In [ ]:
# /eval - Python (60-120 s: 10 questions through the full pipeline)
r = requests.post(f'{BASE}/eval')
print('Status :', r.status_code)
data = r.json()

print()
print('Five-metric dashboard (each row = 0.10):')
for k in ['groundedness', 'citation_validity', 'citation_recall',
          'false_answer_rate', 'false_refusal_rate']:
    print(f'  {k:<22}: {data[k]:.2f}')
print()
print('Rows scored:', data['rows_scored'])
for i, row in enumerate(data['rows'], 1):
    print(f"  #{i:>2}  {'PASS' if row['passed'] else 'FAIL'}  expect={row['expected']:<7} got={row['actual']:<9} {row['question'][:55]}")


---
## 10 · Before/After Comparison - `POST /eval/compare`

The core teaching point of the week: run every golden question through **both** the W5 baseline (hybrid dense+BM25+RRF only) and the full W6 pipeline, sequentially row by row, and return paired metrics.

Note: the baseline is W5's *retrieval* (same collection, embedder, hybrid+RRF over the same narrow pool) **plus W5's threshold gate, re-implemented verbatim** - refuse unless `top1 >= 0.55 and spread >= 0.08`. A "before" column that cannot refuse is not the before: it scores 0.0 on `false_refusal_rate` for free. The full W6 pipeline gates on the **same rule at the same values**, evaluated against both of its channels: `refuse unless (raw_ok or hyde_ok)`.

Expected response shape:
```json
{
  "rows_scored": 10,
  "baseline": { "groundedness": 0.8, "citation_validity": 1.0, "citation_recall": 0.8, "false_answer_rate": 0.0, "false_refusal_rate": 0.2 },
  "full":     { "groundedness": 1.0, "citation_validity": 1.0, "citation_recall": 1.0, "false_answer_rate": 0.0, "false_refusal_rate": 0.0 },
  "rows": [ { "question": "...", "baseline_passed": true, "full_passed": true, "...": "..." } ]
}
```

**There is no row where `baseline_passed` is `true` and `full_passed` is `false`.** Two rows moved, both from baseline-FAIL to W6-PASS:

- **Row #3** (`baseline_passed=false`, `full_passed=true`) - *"How does the decoder prevent a position from using information from later positions when making predictions?"* One of the two rows that moved. Read its four trace floats before you draw a conclusion: `top1_raw=0.579` **clears** the 0.55 threshold, so this was *not* a recall failure - the right chunk was already retrieved. It died on `spread_raw=0.0676` against the 0.08 floor: bunched candidates, i.e. ambiguous. The HyDE channel (`top1_hyde=0.757`, `spread_hyde=0.1796`) cleared both. **HyDE's contribution here is disambiguation, not recall.**
- **Row #5** (`baseline_passed=false`, `full_passed=true`) - the second recovered row. The baseline gate refused it; on this run the HyDE probe cleared the gate and W6 answered and cited - the same gate-recovery mechanism as row #3.

The other eight rows land identically in both columns (rows #9/#10 correctly refuse the off-topic questions). **The entire delta is two questions out of ten - directional, not statistical.** Also note `citation_validity`: it reads 1.0 in both columns because the proxy checks that the model only cited chunks we handed it - a guardrail against a bug, not a quality signal.

Each question runs **twice**, so expect **150-210 s**.


In [ ]:
# /eval/compare - Python (150-210 s: each question runs through BOTH pipelines)
r = requests.post(f'{BASE}/eval/compare')
print('Status :', r.status_code)
data = r.json()

b, f = data['baseline'], data['full']
print()
print(f"{'metric':<22} {'baseline':>9} {'full W6':>9} {'delta':>8}")
for k in ['groundedness', 'citation_validity', 'citation_recall',
          'false_answer_rate', 'false_refusal_rate']:
    print(f'{k:<22} {b[k]:>9.2f} {f[k]:>9.2f} {f[k]-b[k]:>+8.2f}')

print()
print('Per-row paired outcome:')
for i, row in enumerate(data['rows'], 1):
    bp, fp = row['baseline_passed'], row['full_passed']
    tag = 'regressed!' if bp and not fp else ('improved' if fp and not bp else 'same')
    print(f"  #{i:>2}  B {'PASS' if bp else 'FAIL'} | W6 {'PASS' if fp else 'FAIL'}  {tag:<10} {row['question'][:50]}")


---
## 11 · Failure Mode - Retrieval Returns Nothing (503)

When the Qdrant collection is empty or unreachable, the retrieval pipeline returns an empty list. The `/ask` route detects this and returns **503 Service Unavailable** rather than hallucinating an unsupported answer.

This is the same behaviour as W5 CitationRAG - the contract is preserved even though the retrieval path changed.

The browser UI shows a contextual explanation (blue notice box) pointing at the `QDRANT_*` settings when this happens, rather than a generic error.


In [ ]:
# Point QDRANT_COLLECTION at an empty collection (or stop Qdrant) to see the guardrail fire.
# With a populated W4 index this returns 200 and a grounded answer instead.
r = requests.post(f'{BASE}/ask', json={'query': DEMO_QUERY})
print(f'Status: {r.status_code}  (503 = guardrail fired, retrieval returned no candidates)')
print(json.dumps(r.json(), indent=2))


---
## 12 · Failure Mode - Empty Query (422)

Pydantic validates the request body before any retrieval or model call. `QueryRequest.query` has `min_length=1`, so an empty string is rejected immediately - **no tokens spent**.

In [ ]:
%%cmd
curl -s -X POST http://localhost:8000/ask -H "Content-Type: application/json" -d "{\"query\": \"\"}"

In [ ]:
r = requests.post(f'{BASE}/ask', json={'query': ''})
print(f'Status: {r.status_code}  (expected 422 -- Pydantic rejects empty string, no API call made)')
print(json.dumps(r.json(), indent=2))

---
## 13 · Failure Mode - Compressor Marker Stripping (the invariant)

This is **Failure #3** - the one that breaks first when compression is naive, and then gets fixed.

**What breaks without the fix:** the chunk marker (`[chunk-id]`) scores low against the query (it contains no query keywords). A naive compressor drops it. Every downstream citation lookup fails - the citation validator can no longer find the chunk.

**The fix in `compressor.py`:** split the text into `marker + body` *before* scoring. Only body sentences are eligible for removal. The marker is anchored and always written back.

The smoke test `test_compressor_preserves_marker_line` enforces this invariant in CI.

In [ ]:
from app.compressor import compress_chunk
from app.schemas import Chunk

# This chunk's marker has zero lexical overlap with the query.
# Without the fix, it would be scored and dropped.
chunk = Chunk(
    chunk_id='runbook-jwt-rotation-042',
    text=(
        '[runbook-jwt-rotation-042]\n'
        'An apple a day keeps the doctor away. '
        'The JWT signing key lives in the secrets manager. '
        'Cats are good pets. '
        'Cutover happens during the overlap window.'
    ),
    source='runbook.md',
)

compressed = compress_chunk(chunk, query='how do I rotate the JWT signing key')

# Invariant: marker must survive regardless of query overlap
marker_survived = compressed.marker_line == '[runbook-jwt-rotation-042]'
print('Marker survived compression :', marker_survived)
print('Relevant body sentence kept :', 'secrets manager' in compressed.text)
print()
print('Compressed text:')
print(compressed.text)

---

## 14 &middot; Run Smoke Tests

All 21 tests run without an API key and without a vector index. They cover the wiring contract, the threshold gate and its two channels, the marker-preservation invariant, and the citation metric's ability to fail.

```
tests/test_endpoint.py            5 tests   health, config, the reranker toggle,
                                            and two compressor behaviours
tests/test_gate.py               10 tests   the threshold rule, both channels,
                                            and the symmetric gate
tests/test_citation_validity.py   6 tests   the metric must be able to say no
```

`pytest -q` prints `21 passed`. `pytest -q -k marker` prints `1 passed, 20 deselected`.


In [ ]:
%%cmd
cd /d "%CD%" && pytest tests/ -v

In [ ]:
%%cmd
pytest tests/ -q -k marker

---
## 15 · Guided Lab - the four retrieval-layer metrics

The concept video (V1, [11:30]) ends on the eval-first lens and hands you the lab:

> *"In this week's guided lab, these four retrieval-layer numbers - recall at five, p fifty, p ninety five, tokens per query - are yours to compute against the golden set as the lab extension."*

This is that lab. The five metrics you have seen so far (`groundedness`, `citation_validity`,
`citation_recall`, `false_answer_rate`, `false_refusal_rate`) are **product-layer** metrics -
they grade the answer. The four below are **retrieval-layer** metrics - they grade the
*pipeline that fed* the answer. When a product metric drops, these are the numbers that tell
you whether retrieval or generation is to blame.

| Metric | Definition here | Why it matters |
|---|---|---|
| **recall@5** | fraction of answerable golden rows whose `_source_chunk_id` appears in the final top-5 | if the right chunk never arrives, no prompt can save you |
| **p50 latency** | median wall-clock time of `retrieve()` | what a typical user feels |
| **p95 latency** | 95th-percentile wall-clock time of `retrieve()` | what your angriest user feels - always report the tail |
| **tokens/query** | tokens of context injected into the prompt, per query | the line item nobody tracks until the CFO asks |

**Ground truth for recall@5.** `scripts/build_golden_dataset.py` (Week 5) generated each
answerable question *from a specific indexed chunk* and recorded its id in `_source_chunk_id`.
That chunk is the one the retriever must find. The two refusal rows (`must_cite == []`) have no
source chunk and are excluded from the recall denominator - a refusal row has nothing to recall.

**Prerequisites:** the server does not need to be running - we call `retrieve()` in-process -
but `QDRANT_*` and `OPENAI_API_KEY` must be set in `.env` (HyDE makes one nano call per query).
Budget ~1-2 minutes for the 10 rows.


In [ ]:
# --- Lab: measure the retrieval layer ------------------------------------------
# Runs the FULL W6 pipeline (HyDE -> hybrid -> union(30) -> rerank -> top-5 -> compress)
# once per golden row, timing each call and recording what came back.

import json, time, statistics, os, sys
sys.path.insert(0, os.getcwd())

from app.retriever import retrieve, retrieve_baseline
from app.config import get_settings

settings = get_settings()
golden = json.load(open(settings.golden_dataset_path))

TOP_K = settings.vector_top_k_narrow          # 5
CHARS_PER_TOKEN = 4                           # cheap, model-agnostic estimate

def estimate_tokens(chunks) -> int:
    """Approximate tokens injected into the prompt: ~4 chars per token.

    Swap in tiktoken for an exact count - the point of the metric is the trend,
    not the third decimal place."""
    return sum(len(c.text) for c in chunks) // CHARS_PER_TOKEN

def measure(retrieve_fn, rows, label):
    latencies_ms, token_counts, hits, answerable = [], [], 0, 0
    for row in rows:
        t0 = time.perf_counter()
        chunks = retrieve_fn(row['question'])
        latencies_ms.append((time.perf_counter() - t0) * 1000)
        token_counts.append(estimate_tokens(chunks))

        gold = row.get('_source_chunk_id')
        if gold:                                    # refusal rows have none
            answerable += 1
            if gold in {c.chunk_id for c in chunks[:TOP_K]}:
                hits += 1

    latencies_ms.sort()
    def pct(p):
        i = min(int(round(p / 100 * len(latencies_ms))) - 1, len(latencies_ms) - 1)
        return latencies_ms[max(i, 0)]

    return {
        'pipeline':      label,
        'recall@5':      round(hits / answerable, 3) if answerable else float('nan'),
        'p50_ms':        round(statistics.median(latencies_ms)),
        'p95_ms':        round(pct(95)),
        'tokens/query':  round(statistics.mean(token_counts)),
        'rows':          len(rows),
        'answerable':    answerable,
    }

full = measure(retrieve, golden, 'full W6')
print(json.dumps(full, indent=2))


In [ ]:
# --- Lab, part 2: is the W6 pipeline actually earning its latency? --------------
# Same four metrics against the Week 5 baseline (hybrid + RRF only, no HyDE,
# no rerank, no compression). This is the comparison that decides whether the
# extra model call and the cross-encoder pay for themselves ON YOUR CORPUS.

baseline = measure(retrieve_baseline, golden, 'W5 baseline')

hdr = f"{'pipeline':<14}{'recall@5':>10}{'p50 ms':>9}{'p95 ms':>9}{'tokens/query':>15}"
print(hdr)
print('-' * len(hdr))
for m in (baseline, full):
    print(f"{m['pipeline']:<14}{m['recall@5']:>10}{m['p50_ms']:>9}{m['p95_ms']:>9}{m['tokens/query']:>15}")

d_recall = full['recall@5'] - baseline['recall@5']
d_p95    = full['p95_ms']   - baseline['p95_ms']
d_tokens = full['tokens/query'] - baseline['tokens/query']
print()
print(f'delta: recall@5 {d_recall:+.3f} - p95 {d_p95:+.0f} ms - tokens/query {d_tokens:+.0f}')
print()
print('Read it like an engineer, not a fan:')
print('  - recall@5 up and p95 inside your budget  -> the pipeline earned its keep.')
print('  - recall@5 flat and p95 up                -> you bought latency and got nothing.')
print('  - tokens/query down at equal recall       -> compression is doing real work.')


### What to do with these four numbers

1. **Write them down before you change anything.** A retrieval change with no before-number is a
   guess, not an experiment.
2. **Ten rows is a resolution of 0.10 on recall@5** - a swing of one row. Do not report a
   "10-point improvement" off this golden set. Grow the set before you grow the claim; Week 7
   (BreakRAG) makes that argument with a power calculation.
3. **Latency here is measured in-process with a warm reranker.** Your first call pays the
   `bge-reranker-base` model load (~1 GB download on the very first run, then ~2 s from disk
   cache). Discard the first row, or call `retrieve()` once before you start timing, if you
   want a clean p50.
4. **Track `tokens/query` from day one.** It is the only one of the four that turns up in a
   finance meeting.

**Extensions, in order of value:**
- Set `HYDE_ENABLED=false` in `.env` and re-run. Did recall@5 move? On the *Attention* paper -
  an academic corpus whose prose is nothing like a user question - it usually does. On a corpus
  already written in the user's voice, HyDE can make retrieval *worse*. Measure; don't assume.
- Sweep `COMPRESSOR_KEEP_FRACTION` (1.00 / 0.80 / 0.50) and plot tokens/query against
  groundedness from `/eval`. Find the knee.
- Swap `estimate_tokens` for a real `tiktoken` count and see how far the 4-chars-per-token
  rule of thumb was off.


---
## 16 · OpenAPI / Swagger Docs
FastAPI auto-generates interactive docs - try endpoints live in the browser:

> This section is identical across all weeks. Do not modify it.

In [ ]:
from IPython.display import display, HTML
display(HTML('<a href="http://localhost:8000/docs" target="_blank" style="font-size:15px">'
             'Open Swagger UI: http://localhost:8000/docs</a>'))